In [ ]:
import cv2
import os
import numpy as np
from itungorang import PoseSwitcher
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay


In [ ]:
#Feature Extraction and Dataset Creation
SEQ_LEN = 24

def process_clip(path):
    cap = cv2.VideoCapture(path)
    switcher = PoseSwitcher()
    seq = []
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        _, people = switcher.get_people(frame)
        if people:
            seq.append(people[0])  
    cap.release()
    return np.array(seq)

def resample(seq, target_len=SEQ_LEN):
    if len(seq) == 0:
        return None
    idx = np.linspace(0, len(seq) - 1, target_len).astype(int)
    return seq[idx]

def build_dataset(data_dir="data", out_path="extracted_features/dataset.npz"):
    X, y = [], []
    classes = sorted(d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d)))
    for label in classes:
        label_dir = os.path.join(data_dir, label)
        for fname in os.listdir(label_dir):
            if not fname.lower().endswith((".mp4", ".mov", ".avi")):
                continue
            seq = process_clip(os.path.join(label_dir, fname))
            fixed = resample(seq)
            if fixed is not None:
                X.append(fixed); y.append(label)
                print(f"{label}/{fname} -> {fixed.shape}")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    np.savez(out_path, X=np.array(X), y=np.array(y), classes=classes)
    print(f"Dataset saved: X={np.array(X).shape}")

if __name__ == "__main__":
    build_dataset()

In [ ]:
os.makedirs("model", exist_ok=True)

data = np.load("extracted_features/dataset.npz", allow_pickle=True)
X, y = data["X"], data["y"]

le = LabelEncoder()
y_enc = le.fit_transform(y)
num_classes = len(le.classes_)
print("Classes:", le.classes_)

# 70% train, 15% val,  15% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_enc, test_size=0.15, random_state=42, stratify=y_enc)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp)

model = models.Sequential([
    layers.Input(shape=(X.shape[1], X.shape[2])),
    layers.LSTM(128, return_sequences=True),
    layers.Dropout(0.3),
    layers.LSTM(64),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dense(num_classes, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

es = tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                     epochs=80, batch_size=16, callbacks=[es])
model.save("model/boxing_lstm.keras")
np.save("model/classes.npy", le.classes_)
np.savez("model/test_split.npz", X_test=X_test, y_test=y_test)
np.savez("model/history.npz",
          accuracy=history.history["accuracy"],
          val_accuracy=history.history["val_accuracy"],
          loss=history.history["loss"],
          val_loss=history.history["val_loss"])

# grafik hasuil trainingnya
epochs_range = range(1, len(history.history["accuracy"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(epochs_range, history.history["accuracy"], label="Train")
axes[0].plot(epochs_range, history.history["val_accuracy"], label="Validation")
axes[0].set_title("Accuracy over epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].plot(epochs_range, history.history["loss"], label="Train")
axes[1].plot(epochs_range, history.history["val_loss"], label="Validation")
axes[1].set_title("Loss over epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("model/training_curves.png", dpi=150)
plt.show()

print("Training selesai, grafik disimpan di model/training_curves.png")

In [ ]:
#evaluasi dan testing model

model = tf.keras.models.load_model("model/boxing_lstm.keras")
classes = np.load("model/classes.npy", allow_pickle=True)
split = np.load("model/test_split.npz", allow_pickle=True)
X_test, y_test = split["X_test"], split["y_test"]

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}   Test accuracy: {test_acc:.4f}")

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

print("\n=== Test report ===")
print(classification_report(y_test, y_pred, target_names=classes, zero_division=0))

cm = confusion_matrix(y_test, y_pred)


# grafik confusion matrix
fig, ax = plt.subplots(figsize=(6, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
plt.title(f"Confusion Matrix (test accuracy: {test_acc:.2%})")
plt.tight_layout()
plt.savefig("model/confusion_matrix.png", dpi=150)
plt.show()

# grafik perbandingan kurva training dan test
try:
    hist = np.load("model/history.npz")
    epochs_range = range(1, len(hist["accuracy"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(epochs_range, hist["accuracy"], label="Train")
    axes[0].plot(epochs_range, hist["val_accuracy"], label="Validation")
    axes[0].axhline(test_acc, color="red", linestyle="--", label=f"Test ({test_acc:.2f})")
    axes[0].set_title("Accuracy over epochs")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    axes[1].plot(epochs_range, hist["loss"], label="Train")
    axes[1].plot(epochs_range, hist["val_loss"], label="Validation")
    axes[1].axhline(test_loss, color="red", linestyle="--", label=f"Test ({test_loss:.2f})")
    axes[1].set_title("Loss over epochs")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("model/test_vs_training_curves.png", dpi=150)
    plt.show()
except FileNotFoundError:
    print("model/history.npz not found.")